# 02 - Modeling
Alur: bandingkan model (CV Macro F1) -> tuning model terbaik -> CV final + bobot kelas -> evaluasi -> feature importance -> simpan model.

Semua langkah memakai `Pipeline` + `ColumnTransformer` dari `scripts/`. Neural Network **tidak** dipakai.

In [ ]:
import sys
import json
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

sys.path.insert(0, "..")      # agar folder scripts/ bisa di-import dari notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

from scripts.preprocess import load_data, split_xy
from scripts.train import (
    MODEL_NAMES, build_pipeline, cross_validate_macro_f1,
    optimize_class_weights, save_pipeline, subsample, tune_pipeline,
)
from scripts.visualize import (
    plot_actual_vs_predicted, plot_confusion_matrix, plot_feature_importance,
    plot_model_comparison, plot_per_class_metrics,
)

# ---- lokasi data dan output (otomatis: Kaggle atau lokal) ----
ON_KAGGLE = Path("/kaggle/input").exists()
if ON_KAGGLE:
    DATA_DIR = next(Path("/kaggle/input").glob("**/train.csv")).parent
    OUT_DIR = Path("/kaggle/working")
else:                                   # lokal: csv di ../data, output ke root repo
    DATA_DIR = Path("../data")
    OUT_DIR = Path("..")
print("DATA_DIR:", DATA_DIR, "| OUT_DIR:", OUT_DIR)

# ---- parameter eksperimen ----
SEED = 42
SAMPLE_N = 100_000    # subset untuk membandingkan model dan tuning (agar cepat)
TUNE_ITER = 12        # jumlah kombinasi RandomizedSearchCV
CV_SPLITS = 5         # fold CV final pada seluruh data

## 1. Memuat data

In [ ]:
train, test, sample = load_data(DATA_DIR)
print("train:", train.shape, "| test:", test.shape, "| sample:", sample.shape)
X, y = split_xy(train)
train.head()

## 2. Perbandingan model (subset data, 3-fold)

In [ ]:
X_s, y_s = subsample(X, y, SAMPLE_N, SEED)
rows = []
for name in MODEL_NAMES:
    print(name)
    res = cross_validate_macro_f1(build_pipeline(name, SEED), X_s, y_s, n_splits=3, seed=SEED)
    rows.append({"model": name, "macro_f1_mean": res["mean"], "macro_f1_std": res["std"]})
comparison = pd.DataFrame(rows).set_index("model").sort_values("macro_f1_mean", ascending=False)
BEST_MODEL = comparison.index[0]
print("Model terbaik:", BEST_MODEL)
plot_model_comparison(comparison)
comparison

## 3. Tuning model terbaik (`RandomizedSearchCV`, di dalam pipeline)

In [ ]:
tuned, best_params, tune_score = tune_pipeline(
    build_pipeline(BEST_MODEL, SEED), X_s, y_s, BEST_MODEL, n_iter=TUNE_ITER, seed=SEED
)
print("Param terbaik:", best_params)
print(f"Macro F1 CV (tuning): {tune_score:.4f}")

## 4. CV pada seluruh data + optimasi bobot kelas
Bobot dicari pada probabilitas *out-of-fold*, lalu dipakai saat prediksi test.

In [ ]:
cv = cross_validate_macro_f1(tuned, X, y, n_splits=CV_SPLITS, seed=SEED)
print(f"Macro F1 CV {CV_SPLITS}-fold: {cv['mean']:.4f} +/- {cv['std']:.4f}")

weights, weighted_score = optimize_class_weights(cv["oof_proba"], y, cv["classes"])
print("Bobot kelas:", np.round(weights, 2).tolist())
print(f"Macro F1 OOF dengan bobot: {weighted_score:.4f}")

pred_oof = cv["classes"][(cv["oof_proba"] * weights).argmax(axis=1)]
print(classification_report(y, pred_oof, digits=3))

## 5. Evaluasi: confusion matrix, metrik per kelas, distribusi prediksi

In [ ]:
plot_confusion_matrix(y, pred_oof)
plot_per_class_metrics(y, pred_oof)
plot_actual_vs_predicted(y, pred_oof)

## 6. Feature importance (permutation importance pada hold-out)

In [ ]:
imp = plot_feature_importance(tuned, X_s, y_s, seed=SEED)
imp.sort_values(ascending=False).head(10)

## 7. Fit akhir dan simpan model

In [ ]:
final_model = tuned.fit(X, y)
model_path = save_pipeline(final_model, BEST_MODEL, weights)
print("Model tersimpan:", model_path)

REPORT_DIR = OUT_DIR / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
summary = {
    "model": BEST_MODEL,
    "best_params": {k: (v.item() if hasattr(v, "item") else v) for k, v in best_params.items()},
    "cv_folds": CV_SPLITS,
    "cv_macro_f1_mean": cv["mean"],
    "cv_macro_f1_std": cv["std"],
    "macro_f1_oof_with_class_weights": weighted_score,
    "class_weights": [float(w) for w in weights],
}
(REPORT_DIR / f"cv_{BEST_MODEL}.json").write_text(json.dumps(summary, indent=2))
summary

## 8. Interpretasi (isi setelah menjalankan notebook)
- Model terbaik dan alasannya: ...
- Pengaruh tuning dan bobot kelas terhadap Macro F1: ...
- Kelas yang paling sulit diprediksi dan pola kesalahan pada confusion matrix: ...
- Fitur paling berpengaruh dan penjelasan rekayasanya (mengapa masuk akal secara struktural): ...